## Deep Research

A **multi-agent pipeline** that runs a full research workflow from a single query: it asks clarification questions, plans web searches, runs them, then writes a long-form report and (optionally) emails it via Resend.

**Flow:** Manager → **Clarifier** (narrow the query) → **Planner** (search terms) → **Search** (web, once per term) → **Writer** (outline + report) → **Email** (HTML report to your inbox).

**When the handoff happens:** The Manager keeps control until _all_ web searches are done. Only then does it **hand off** to the Writer agent with the gathered research. The Writer then owns outline → report → email and does not hand back to the Manager.


In [ ]:
# --- Setup ---
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
import resend
from typing import Dict

load_dotenv(override=True)

In [ ]:
# Config (tunable)
HOW_MANY_SEARCHES = 2
HOW_MANY_CLARIFICATION_QUESTIONS = 3

## Pydantic schemas (structured outputs)

All response shapes used by the agents.


In [ ]:
# Clarification questions
class ClarificationQuestionItems(BaseModel):
    reason: str = Field(description="Your reasoning for why this question is important to the query.")
    question: str = Field(description="The question to ask to get a detailed answer for the query.")


class ClarificationQuestions(BaseModel):
    questions: list[ClarificationQuestionItems] = Field(
        description="A list of clarification questions to ask for the query."
    )


# Web search plan
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query."
    )


# Final report
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report.")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further.")

## Search agent

Searches the web and returns a concise summary for a given query.


In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)
search_tool = search_agent.as_tool(tool_name="search_tool",tool_description="Search the web for a query")

## Clarification & planner agents

Generate clarification questions and a web-search plan using structured outputs.


In [ ]:
# Planner: outputs a list of web search queries (uses WebSearchPlan from schemas above)
INSTRUCTIONS = (
    f"You are a helpful research assistant. Given a set of questions, come up with web searches "
    f"to perform to best answer them. Output {HOW_MANY_SEARCHES} terms to query for."
)
planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)


In [ ]:
# Clarification agent (uses ClarificationQuestions from schemas above)
INSTRUCTIONS = (
    f"You are a senior researcher. Given a query, come up with clarification questions "
    f"to get a detailed answer. Output {HOW_MANY_CLARIFICATION_QUESTIONS} questions."
)
clarification_agent = Agent(
    name="ClarificationAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ClarificationQuestions,
)

In [ ]:
# Expose clarifier and planner as tools for the manager
clarifier_tool = clarification_agent.as_tool(
    tool_name="clarifier_tool",
    tool_description="Generate clarification questions for a query",
)
planner_tool = planner_agent.as_tool(
    tool_name="planner_tool",
    tool_description="Generate web searches for clarification questions from the clarifier.",
)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    resend.api_key = os.environ.get("RESEND_API_KEY")
    if not resend.api_key:
        return {"error": "RESEND_API_KEY not set"}
    params = {
        "from": "onboarding@resend.dev", 
        "to": ["williamikeji@gmail.com"],
        "subject": subject,
        "html": html_body,
    }
    resend.Emails.send(params)
    return {"status": "success"}

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [ ]:
email_tool = email_agent.as_tool(tool_name="email_tool",tool_description="Send an email with the report converted into clean, well presented HTML with an appropriate subject line.")
reporter_tools = [email_tool]

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher assistant. You receive the original query and research from the team. "
    "First, create an outline for the report (structure and flow). "
    "Then write the full report in markdown: detailed, 1000+ words, 5–10 pages of content. "
    "Finally, use the email tool to send the report: convert your markdown to clean, well-formatted HTML "
    "for the email body and choose a clear subject line. Send exactly one email with the full report."
)


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    tools=reporter_tools,
    output_type=ReportData,
    handoff_description="Write the report and send it to the recipient using the email tool."
)

In [ ]:
# Manager: clarifier once → planner once → search exactly HOW_MANY_SEARCHES times → handoff
INSTRUCTIONS = (
    "You are a research lab manager. You will be provided with a query. Follow this sequence exactly:\n"
    "1. Call clarifier_tool **once** with the query. You get back clarification questions.\n"
    "2. Call planner_tool **once** with those questions. You get back exactly " + str(HOW_MANY_SEARCHES) + " search terms.\n"
    "3. Call search_tool **exactly " + str(HOW_MANY_SEARCHES) + " times** — once per search term from the planner. "
    "Do not call the planner again. Do not call search_tool more than " + str(HOW_MANY_SEARCHES) + " times.\n"
    "4. After you have exactly " + str(HOW_MANY_SEARCHES) + " search results, hand off to the writer agent once. "
    "Do not hand off before all searches are done."
)

manager_tools = [clarifier_tool, planner_tool, search_tool]

manager_agent = Agent(
    name="ManagerAgent",
    instructions=INSTRUCTIONS,
    tools=manager_tools,
    model="gpt-4o-mini",
    handoffs=[writer_agent]
)


In [ ]:
query = "Latest AI Agent frameworks in 2025"

# Deep research needs many turns: clarifier → planner → N searches → writer → email
with trace("Research trace"):
    await Runner.run(manager_agent, query, max_turns=30)
    print("Hooray!")




## Trace

https://platform.openai.com/traces
